# 🎙️ AK-AIGC-Studio: CosyVoice 2 快速起步实验手册
> **主理人**: B1ueAkisa | **AI 总副手**: Gemini (Google DeepMind)

本 Notebook 将帮助你一键完成：
1. 挂载 Google Drive，将模型缓存和输出结果永久存放在 `AIGC_Workspace`；
2. 自动拉取阿里开源的 **CosyVoice2-0.5B** 旗舰多语言/零样本声音克隆模型；
3. 体验高自然度 Zero-Shot 声音克隆与四川话/粤语等方言指令合成，并直接试听导出！

### Step 1: 检查 GPU 状态与挂载 Google Drive

In [ ]:
# 检查显卡状态
!nvidia-smi

# 挂载 Google Drive 工作区
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("提示 (非 Colab 或已挂载):", e)

WORKSPACE = '/content/drive/MyDrive/AIGC_Workspace'
MODELS_AUDIO_DIR = os.path.join(WORKSPACE, 'models/audio')
OUTPUTS_DIR = os.path.join(WORKSPACE, 'outputs')
os.makedirs(MODELS_AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
print(f"✅ Google Drive 工作区就绪: {WORKSPACE}")

### Step 2: 克隆 CosyVoice 官方库并安装轻量化推理依赖 (防卡死 & 编译优化)

In [ ]:
# [1/3] 安装底层音频编解码系统依赖 (避免 sox/torchaudio 动态库报错)
print("📦 [1/3] 安装底层音频编解码系统依赖 (sox, libsox-dev, ffmpeg)...")
!apt-get update -qq && apt-get install -y -qq sox libsox-dev ffmpeg

# [2/3] 浅克隆 CosyVoice 仓库并确保完整检出 Matcha-TTS 子模块
print("🐙 [2/3] 克隆 CosyVoice 仓库及 Matcha-TTS 子模块...")
import os
if not os.path.exists('/content/CosyVoice'):
    !git clone --depth 1 --recursive https://github.com/FunAudioLLM/CosyVoice.git /content/CosyVoice
else:
    !cd /content/CosyVoice && git submodule update --init --recursive

# [3/3] 依赖优化：剔除 deepspeed/tensorrt 等训练级重型编译库(防止 Colab 源码编译卡死)，复用系统 torch
print("⚡ [3/3] 安装轻量化推理依赖...")
%cd /content/CosyVoice
!sed -i -E '/deepspeed|tensorrt|aiinfra\.pkgs\.visualstudio/d' requirements.txt
!sed -i -E 's/torch==2\.3\.1/torch>=2.1.0/g; s/torchaudio==2\.3\.1/torchaudio>=2.1.0/g' requirements.txt
!pip install -r requirements.txt
!pip install modelscope sox
print("✅ 环境依赖就绪！")

### Step 3: 下载 / 读取 CosyVoice 2 模型 (智能缓存检测，秒级加载)

In [ ]:
import os
import sys
sys.path.append('/content/CosyVoice')
sys.path.append('/content/CosyVoice/third_party/Matcha-TTS')

from modelscope import snapshot_download

MODEL_DIR = '/content/drive/MyDrive/AIGC_Workspace/models/audio/CosyVoice2-0.5B'

# 检查云盘中是否已存在核心权重文件 (若已存在则直接秒级复用，避免重复下载)
required_weights = ['llm.pt', 'flow.pt', 'hift.pt', 'cosyvoice2.yaml']
is_ready = os.path.exists(MODEL_DIR) and all(os.path.exists(os.path.join(MODEL_DIR, f)) for f in required_weights)

if is_ready:
    print(f"⚡ 检测到 Google Drive 中已存在完整 CosyVoice2 权重，直接复用: {MODEL_DIR}")
else:
    print(f"📦 正在从 ModelScope 下载 CosyVoice2-0.5B 到: {MODEL_DIR} ...")
    snapshot_download('iic/CosyVoice2-0.5B', local_dir=MODEL_DIR)
    print("✅ 模型下载完成并已持久化到 Google Drive！")

### Step 4: 加载 CosyVoice 2 模型并体验旗舰 Zero-Shot 声音克隆

In [ ]:
import os
import torch
import torchaudio
from IPython.display import Audio, display
from cosyvoice.cli.cosyvoice import CosyVoice2

print("🚀 正在初始化 CosyVoice2 模型并载入显存 (FP16)...\n")
cosyvoice = CosyVoice2(MODEL_DIR, load_jit=False, load_trt=False, fp16=True)
if torch.cuda.is_available():
    print("✅ 模型加载成功！显存已用: {:.2f} GB\n".format(torch.cuda.memory_allocated() / 1e9))
else:
    print("✅ 模型已在 CPU 模式加载！\n")

# CosyVoice2 零样本声音克隆演示
prompt_wav = '/content/CosyVoice/asset/zero_shot_prompt.wav'
prompt_text = '希望你以后能够做的比我还好呦。'
text = '欢迎来到 AK-AIGC-Studio！这里是你的专属多模态生成工坊，由主理人 B1ueAkisa 与总副手 Gemini 共同打造。'

print(f"🎙️ 开始 Zero-Shot 语音合成: {text}")
output_wav = os.path.join(OUTPUTS_DIR, 'welcome_gemini.wav')
for i, j in enumerate(cosyvoice.inference_zero_shot(text, prompt_text, prompt_wav)):
    torchaudio.save(output_wav, j['tts_speech'], cosyvoice.sample_rate)
    break

print(f"🎉 合成完成！已自动存入 Google Drive: {output_wav}")
display(Audio(output_wav))

### Step 5: (进阶体验) CosyVoice 2 方言与情感指令控制 (Instruct Mode)

In [ ]:
# 体验 CosyVoice2 强大的方言控制 (如四川话、粤语等) 与指令跟随
dialect_instruct = '用四川话说这句话<|endofprompt|>'
dialect_text = '今天天气巴适得很，咱们 AK-AIGC-Studio 的语音大模型跑起来咯！'

print(f"🎙️ 正在生成四川方言音频: {dialect_text}")
dialect_wav = os.path.join(OUTPUTS_DIR, 'dialect_sichuan.wav')
for i, j in enumerate(cosyvoice.inference_instruct2(dialect_text, dialect_instruct, prompt_wav)):
    torchaudio.save(dialect_wav, j['tts_speech'], cosyvoice.sample_rate)
    break

print(f"🎉 方言合成完成！已存入云盘: {dialect_wav}")
display(Audio(dialect_wav))